<a href="https://colab.research.google.com/github/JustLikeTech/CycleGAN_Rebuild/blob/main/testprogram_mbakdini(full).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix

Cloning into 'pytorch-CycleGAN-and-pix2pix'...
remote: Enumerating objects: 2619, done.
remote: Total 2619 (delta 0), reused 0 (delta 0), pack-reused 2619 (from 1)
Receiving objects: 100% (2619/2619), 8.23 MiB | 16.83 MiB/s, done.
Resolving deltas: 100% (1654/1654), done.


In [ ]:
import os
print(os.listdir())  # Menampilkan semua folder dan file di direktori saat ini

['.config', 'drive', 'pytorch-CycleGAN-and-pix2pix', 'sample_data']


In [ ]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
!ls -F

drive/	pytorch-CycleGAN-and-pix2pix/  sample_data/


In [ ]:
# DOMINATE?
# A Python library for automatically generating HTML pages,
# typically used in projects like CycleGAN to display training results.

In [ ]:
!pip install dominate
import dominate
print("dominate module installed successfully")

dominate module installed successfully


In [ ]:
import wandb
!wandb login 390063a1a72a99da508afc81c6b099a74c5c6afc

wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## Roboflow Dataset

In [ ]:
!pip install roboflow

import roboflow
# Inisialisasi API Roboflow
dataset_name = "traintestrgbndvi-ilx7x"
dataset_ver = 1
rf = roboflow.Roboflow(api_key="uws7Wv4LJKXm3PrYyg9a")
project = rf.workspace("datasetta-ubg4x").project(dataset_name)
version = project.version(dataset_ver)

# Download dataset dalam format yang sesuai
dataset = version.download("folder", location="/kaggle/working/cyclegan-dataset")

print("Dataset downloaded successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 48.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/cyclegan-dataset in folder:: 100%|██████████| 608/608 [00:00<00:00, 2900.35it/s]

Dataset downloaded successfully!


### Split Data

In [ ]:
import os
from sklearn.model_selection import train_test_split
from shutil import copyfile

# Define dataset name and version
dataset_name = 'cyclegan-dataset'
dataset_ver = '1.0'

# Set the root folder
root_folder = '/kaggle/working/' + dataset_name
input_folder = os.path.join(root_folder, 'train')
# Correct the output_folder path to reflect the actual location of the cloned repository
output_folder = '/content/pytorch-CycleGAN-and-pix2pix/datasets/datasetta-ubg4x'

# Verify input folder
if not os.path.exists(input_folder):
    raise FileNotFoundError(f"Input folder '{input_folder}' not found.")

# Create output folders if they don't exist
for folder_name in ['trainA', 'trainB', 'testA', 'testB']:
    folder_path = os.path.join(output_folder, folder_name)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

# Define paths for ndvi and rgb
ndvi_path = os.path.join(input_folder, 'ndvi')
rgb_path = os.path.join(input_folder, 'rgb')

# Verify subfolders
if not os.path.exists(ndvi_path):
    raise FileNotFoundError(f"'ndvi' folder not found at {ndvi_path}")
if not os.path.exists(rgb_path):
    raise FileNotFoundError(f"'rgb' folder not found at {rgb_path}")

# List all files in the input folder for each category
ndvi_files = [f for f in os.listdir(ndvi_path) if f.endswith('.jpg')]
rgb_files = [f for f in os.listdir(rgb_path) if f.endswith('.jpg')]

# Split the data into training and test sets for both ndvi and rgb
rgb_train, rgb_test, ndvi_train, ndvi_test = train_test_split(rgb_files, ndvi_files, test_size=0.2, random_state=2024)

# Copy files to the corresponding folders
for file_name in ndvi_train:
    source_path = os.path.join(ndvi_path, file_name)
    destination_path = os.path.join(output_folder, 'trainB', file_name)
    copyfile(source_path, destination_path)

for file_name in rgb_train:
    source_path = os.path.join(rgb_path, file_name)
    destination_path = os.path.join(output_folder, 'trainA', file_name)
    copyfile(source_path, destination_path)

for file_name in ndvi_test:
    source_path = os.path.join(ndvi_path, file_name)
    destination_path = os.path.join(output_folder, 'testB', file_name)
    copyfile(source_path, destination_path)

for file_name in rgb_test:
    source_path = os.path.join(rgb_path, file_name)
    destination_path = os.path.join(output_folder, 'testA', file_name)
    copyfile(source_path, destination_path)

print("Data split and saved successfully.")

Data split and saved successfully.


#### CycleGAN Train

#### Train with n_epoch = 100

In [ ]:
exp_name = 'traincyclegan'
!python /content/pytorch-CycleGAN-and-pix2pix/train.py --use_wandb --wandb_project_name='cyclegan' --gan_mode vanilla --load_size 416 --crop_size 416 --preprocess 'none' --dataroot /content/pytorch-CycleGAN-and-pix2pix/datasets/datasetta-ubg4x --name $exp_name --model cycle_gan

----------------- Options ---------------
               batch_size: 1                             
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 416                           	[default: 256]
                 dataroot: /content/pytorch-CycleGAN-and-pix2pix/datasets/datasetta-ubg4x	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
             display_freq: 400                           
          display_winsize: 256                           
                    epoch: latest                        
              epoch_count: 1                             
                 gan_mode: vanilla                       	[default: lsgan]
                init_gain: 0.02                          
                init_type: normal                        
        

In [ ]:
import shutil
import os
import zipfile

main_path = '/content'
exp_name = 'traincyclegan'

# Corrected path for CycleGAN training results (e.g., generated images)
source_folder_results = os.path.join(main_path, 'checkpoints', exp_name, 'web', 'images')
destination_folder = '/kaggle/working/'

# Archive training results if the folder exists
if os.path.exists(source_folder_results):
    shutil.make_archive(os.path.join(destination_folder, 'results'), 'zip', source_folder_results)
    print(f"Zip file created for training results: {os.path.join(destination_folder, 'results.zip')}")
else:
    print(f"Warning: Training results folder not found at {source_folder_results}. Skipping results archiving.")

# Archive checkpoint-related files
source_folder_checkpoints = os.path.join(main_path, 'checkpoints', exp_name)
files_to_zip = [
    'test_opt.txt'
]
zip_file_name = 'checkpoints_test.zip'

if os.path.exists(source_folder_checkpoints):
    with zipfile.ZipFile(os.path.join(destination_folder, zip_file_name), 'w') as zipf:
        for item in files_to_zip:
            source_path = os.path.join(source_folder_checkpoints, item)
            if os.path.exists(source_path):
                zipf.write(source_path, arcname=item)
                print(f"Added {item} to {zip_file_name}.")
            else:
                print(f"Warning: File {source_path} not found. Skipping {item}.")
    print(f"Zip file created: {os.path.join(destination_folder, zip_file_name)}")
else:
    print(f"Warning: Checkpoints folder not found at {source_folder_checkpoints}. Skipping checkpoint archiving.")

In [ ]:
import os
import shutil
import zipfile
from datetime import datetime

main_path = '/content'
exp_name = 'traincyclegan'

# Path untuk source dan destination
source_folder_results = os.path.join(main_path, 'checkpoints', exp_name, 'web', 'images')
destination_folder = '/kaggle/working/'

# Membuat ZIP untuk folder hasil
try:
    result_zip_base_name = os.path.join(destination_folder, 'results_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    if os.path.exists(source_folder_results):
        shutil.make_archive(result_zip_base_name, 'zip', source_folder_results)
        print(f"Results zip file created at: {result_zip_base_name}.zip")
    else:
        print(f"Warning: Training results folder not found at {source_folder_results}. Skipping results archiving.")
except Exception as e:
    print(f"Error creating results zip: {e}")

# Path untuk folder checkpoints
source_folder_checkpoints = os.path.join(main_path, 'checkpoints', exp_name)
files_to_zip = ['test_opt.txt']
zip_file_name = f'checkpoints_test_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'

try:
    # Membuat ZIP untuk file tertentu dari checkpoints
    if os.path.exists(source_folder_checkpoints):
        with zipfile.ZipFile(os.path.join(destination_folder, zip_file_name), 'w') as zipf:
            for item in files_to_zip:
                source_path = os.path.join(source_folder_checkpoints, item)
                # Debug: log setiap file yang diproses
                print(f"Processing file: {source_path}")
                if os.path.exists(source_path):
                    if os.path.isdir(source_path):
                        # Add directory to ZIP
                        for root, dirs, files in os.walk(source_path):
                            for file in files:
                                zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), os.path.join(source_folder_checkpoints, os.pardir)))
                        print(f"Added directory {item} to ZIP")
                    else:
                        # Add file to ZIP
                        zipf.write(source_path, arcname=item)
                        print(f"Added {item} to ZIP")
                else:
                    print(f"File not found: {source_path}")
        print(f"Checkpoints zip file created: {zip_file_name}")
    else:
        print(f"Warning: Checkpoints folder not found at {source_folder_checkpoints}. Skipping checkpoint archiving.")
except Exception as e:
    print(f"Error creating checkpoints zip: {e}")

# Verifikasi isi folder destination
print("Contents of destination folder:")
os.system(f'ls -l {destination_folder}')

### Rerunning Training and Generating `test_opt.txt` with n_epoch=1

I will now rerun the training script. Following the training, I will execute the `test.py` script to generate the `test_opt.txt` file, which was previously reported as missing.

In [ ]:
exp_name = 'traincyclegan'
!python /content/pytorch-CycleGAN-and-pix2pix/train.py \
    --use_wandb --wandb_project_name='cyclegan' \
    --gan_mode vanilla --load_size 416 --crop_size 416 --preprocess 'none' \
    --dataroot /content/pytorch-CycleGAN-and-pix2pix/datasets/datasetta-ubg4x \
    --name $exp_name --model cycle_gan \
    --n_epochs 1 --n_epochs_decay 0 \
    --save_latest_freq 100 --save_by_iter